# Second Model: Forecasting GMM

A fundamental limitation of a standard GMM is that it's a density estimation model. It learns $p(\textbf{x})$ and has no concept of time evolution $p(\textbf{x}_{t+1}|\textbf{x}_t)$ at time-step $t$. Our previous model is good at evaluating whether a current vector of environmental factors and vegetation indices is likely to occur given the ones already observed, but unfortunately it cannot be used in its original form as a forecasting model. We now consider a proposal to extend to forecasting by exploiting the conditional distribution of a multivariate Gaussian.

Eirola and Lendasse (2013) proposed that, making use of **delay embedding** of length $d$ to create $d$-dimensional overlapping rolling windows from a time series, we can fit a GMM to the resulting vectors and forecast future observations using the conditional expectation of a multivariate Gaussian. This allows for the prediction of an entire future horizon simultaneously.

Of course one first has to ask why not just ensemble trees for this? Forecasting is a fundamental supervised learning problem, which has many many different (and well researched) models. The  purpose of introducing forecasting through the GMM is _not_ to replace the state-of-the-art methods or to maximise accuracy, but to investigate how the existing probabilistic model can be extended to forecasting and anomaly detection in a shared framework.

## Libraries

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture 
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from models import NDVIClimatologyGAM, LikelihoodGMM, ForecastingGMM, ForecastingGMMSelector
from data import utils

## Functions

In [36]:
def get_interpolated_df(spectral, timestamps):
    spectral_columns = [
        "NDVI_mean",
        "NDWI_mean",
        "NDRE_mean",
    ]

    df = spectral.merge(timestamps, on="Timestamp", how="right")
    interpolated_df = df[["Timestamp"] + spectral_columns].copy()
    interpolated_df[spectral_columns] = interpolated_df[spectral_columns].interpolate(method="pchip", limit_area="inside").bfill()

    return interpolated_df.dropna()

## Data

In [ ]:
eng_features = [
    # Timestamp for merging
    'Timestamp',

    # engineered features
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    'VW_PC1',  
    #'Shallow_mean',
    'ST_PC1',

    # Date features for plotting
    "shifted_doy",
    "Ag_year",
    'in_season',
    
]

non_spectral_features = [
    'Timestamp',
    'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    'precipitation',
    'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    'soil_temperature_level_4'
]


In [37]:
builder = DatasetBuilder(init_df=test_df[eng_features + non_spectral_features])

gmm_df = (
    builder
    .merge_spectral(grouped_spectral_df[spectral_features])
    .interpolate(val_columns=spectral_features[1:], dropna=True)
    .build()
)

train_mask = (gmm_df["Ag_year"] != 2022) 
test_mask = (gmm_df["Ag_year"] == 2022) 
passthrough_cols = ["doy_sin", "doy_cos"]
drop_cols = ["Timestamp", "shifted_doy", "doy", "GDD", "in_season", "Ag_year"]

scaler_splitter = DatasetScalerSplitter(
    train_mask=train_mask, 
    test_mask=test_mask, 
    passthrough_cols=[],
    drop_cols=drop_cols
)

scaler_splitter.fit(gmm_df)

X_train_df, X_test_df = scaler_splitter.train_test_split_transform(gmm_df)

## Delay Embedding

As previously stated, a GMM operates on vectors rather than sequential observations. Consequently, we transform the time series into _overlapping_ windows of length $d$. If the univariate time series is given by $$\textbf{z}=[z_0,z_1,...,z_{n-1}]$$
and we choose the dimension of embedding as $d$, then _each training sample_ is constructed as $$\textbf{x}_t=[z_t, z_{t+1}, ..., z_{t+d-1}], \quad t=0,1,...,n-d$$
From this we form the new data matrix $X:(n-d+1)\times d$ where the rows are in $\mathbb{R}^d$. 

An important thing to note is that this is for one time series - we consider several related variables. In other words, we embed a multivariate time series, with $m$ variables per each observation $t=0,1,...,n-1$, such that $X\in\mathbb{R}^{(n-d+1)\times(md)}$

In [44]:
d = 6

In [45]:
def delay_embedding(X, d):
    X = np.asarray(X)

    n, m = X.shape

    embedded = np.empty(shape=(n-d+1, m * d), dtype=X.dtype)

    for i in range(n-d+1):
        embedded[i] = X[i:i+d].reshape(-1)

    return embedded

In [46]:
scaler = StandardScaler()

X_num = X.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos", "in_season"], axis=1).copy()


test_mask = X["Ag_year"] == 2022
train_mask = ~test_mask

X_train_raw = X_num[train_mask]
X_test_raw = X_num[test_mask]

scaler.fit(X_train_raw)

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=X_num.columns, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_num.columns, index=X_test_raw.index)

X_train_df = pd.concat([X_train, X.loc[train_mask, ["doy_sin", "doy_cos"]]], axis=1)
X_test_df = pd.concat([X_test, X.loc[test_mask, ["doy_sin", "doy_cos"]]], axis=1)

X_train = delay_embedding(X_train_df, d)
X_test = X_test_df.to_numpy()

---

## Conditional Expectations (i.e. Forecasting)

With our regressor size of $d=24$, we can for instance take the last year's measurements as the _first_ 12 months ($P$, known/given), then calculate the conditional expectation of the _next_ 12 months ($F$, unknown). 

$$E[\textbf{x}^F|\textbf{x}^P]$$

Now since each Gaussian component is partitioned into splits $P, F$, it can be shown that for a single component $k$, the conditional expectation of future values conditioned on past sample $\textbf{x}_i^P$ is given by 

$$\tilde{\textbf{y}}_{ik}=E[\textbf{x}^F_i|\textbf{x}^P_i]=\boldsymbol{\mu}_k^F+\Sigma^{FP}_k(\Sigma^{PP}_k)^{-1}(\textbf{x}_i^P-\boldsymbol{\mu}_k^P)$$




Of course, we don't know which component generated point $i$, so if the posterior probability of sample $\textbf{x}_i^P$ belonging to each component $k$ is given by $t_{ik}$ (calculated during the E-step), then the overall prediction is the weighted average 

$$\hat{\textbf{y}}_i=\sum^K_{k=1}t_{ik}\tilde{\textbf{y}}_{ik}$$

### Training

Instinctively, it might seem like a good idea to select the number of components for the GMM corresponding to minimum BIC / AIC like we did  in the previous model. This is not the approach we follow, since we are only interested in forecasting, not density explanation. This is especially true for delay embedding, since each feature is almost identical and could be well-enough explained by one Gaussian. 

In [47]:
n_components = 4
gmm = GaussianMixture(n_components=n_components, covariance_type='full', n_init=10, reg_covar=1e-3)
gmm.fit(X_train)

props = gmm.weights_
means = gmm.means_
covs = gmm.covariances_
covs.shape

(4, 78, 78)

In [48]:
props

array([0.06498503, 0.2488033 , 0.43251912, 0.25369256])

### Prediction

Pretend the last 12 observations of `X_test` are unknown and are to be predicted. Since we trained the GMM on $d=24$, the past $12$ observations are known (denoted by $P$) and the future 12 are (assumed) unknown (denoted by $F$). We form a vector `pred_window` that represents a single window, partitioned into the last 12 known observations and the future 12 set as NaN.

Now since the original paper is restricted to a univariate case, we take some creative liberty in the extension to multivariate. In the above `pred_window`, $P\in\mathbb{R}^{144}$ (12 observations and 12 features). A natural extension would be to let $F\in\mathbb{R}^{144}$ and predict the values of each feature. This is unnecessary, as we are not interested in (nor capable of)  accurately forecasting the solar radiation or the precipitation. What would be beneficial however is forecasting the vegetation indices $\text{NDVI}, \ \text{NDWI}, \text{ and } \text{NDRE}$, as these give an interpretable and reliable representation of water and vegatative stress. This means that we will have 3 features and 12 values, so $F\in\mathbb{R}^{36}$.

Next we iterate through each Gaussian component, partition according to past/future, calculate the expectations and then the weighted average. Note we can't use `gmm.weights_` for $t_{ik}$ since that represents the global probability of each component across the entire dataset (priors), while $t_{ik}$ represents the specific probability of a component _given_ the past data (posterior probabilities). 

# References

Eirola, E. and Lendasse, A. (2013). Gaussian Mixture Models for Time Series Modelling, Forecasting, and Interpolation. [online] Available at: https://research.cs.aalto.fi/aml/Publications/Publication204.pdf